In [1]:

import re
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from rapidfuzz import fuzz

from sklearn.model_selection import train_test_split

tqdm.pandas()

c:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

DATA_DIR = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\processed"
)

TRAIN_PATH = DATA_DIR / "train_preprocessed.csv"
TEST_PATH = DATA_DIR / "test_preprocessed.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

C:\Users\bimal\AppData\Local\Temp\ipykernel_8060\1714900633.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(TEST_PATH)


Training dataset: (404290, 6)
Testing dataset: (3563475, 3)


In [3]:

TARGET = "is_duplicate"
QUESTION_COLUMNS = ["question1", "question2"]

X = train_df[QUESTION_COLUMNS].fillna("")
y = train_df[TARGET].astype("int8")

X_test = test_df[QUESTION_COLUMNS].fillna("")

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))
print("Test samples:", len(X_test))

Training samples: 323432
Validation samples: 80858
Test samples: 3563475


In [4]:

def basic_features(q1, q2):
    q1 = q1 if isinstance(q1, str) else ""
    q2 = q2 if isinstance(q2, str) else ""

    words1 = q1.split()
    words2 = q2.split()

    len1 = len(q1)
    len2 = len(q2)
    count1 = len(words1)
    count2 = len(words2)

    return {
        "q1_char_count": len1,
        "q2_char_count": len2,
        "char_count_diff": abs(len1 - len2),
        "q1_word_count": count1,
        "q2_word_count": count2,
        "word_count_diff": abs(count1 - count2),
        "char_length_ratio": (
            min(len1, len2) / max(len1, len2)
            if max(len1, len2) else 0
        ),
        "word_length_ratio": (
            min(count1, count2) / max(count1, count2)
            if max(count1, count2) else 0
        )
    }

In [5]:

def overlap_features(q1, q2):
    words1 = set(q1.split())
    words2 = set(q2.split())

    common = words1 & words2
    total = words1 | words2

    return {
        "common_word_count": len(common),
        "unique_word_count_q1": len(words1),
        "unique_word_count_q2": len(words2),
        "word_jaccard_similarity": (
            len(common) / len(total) if total else 0
        ),
        "common_word_ratio_q1": (
            len(common) / len(words1) if words1 else 0
        ),
        "common_word_ratio_q2": (
            len(common) / len(words2) if words2 else 0
        )
    }

In [6]:

def fuzzy_features(q1, q2):
    return {
        "fuzzy_ratio": fuzz.ratio(q1, q2),
        "token_sort_ratio": fuzz.token_sort_ratio(q1, q2),
        "token_set_ratio": fuzz.token_set_ratio(q1, q2)
    }

In [7]:

def extract_pair_features(q1, q2):
    features = {}

    features.update(basic_features(q1, q2))
    features.update(overlap_features(q1, q2))
    features.update(fuzzy_features(q1, q2))

    return features


def create_handcrafted_features(data):
    rows = []

    for q1, q2 in tqdm(
        zip(data["question1"], data["question2"]),
        total=len(data),
        desc="Extracting features"
    ):
        rows.append(extract_pair_features(q1, q2))

    return pd.DataFrame(rows, index=data.index, dtype=np.float32)


X_train_hand = create_handcrafted_features(X_train)
X_valid_hand = create_handcrafted_features(X_valid)
X_test_hand = create_handcrafted_features(X_test)

print("Training features:", X_train_hand.shape)
print("Validation features:", X_valid_hand.shape)
print("Test features:", X_test_hand.shape)

Extracting features: 100%|██████████| 3563475/3563475 [00:45<00:00, 77738.48it/s]


Training features: (323432, 17)
Validation features: (80858, 17)
Test features: (3563475, 17)


In [8]:

FEATURE_DIR = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\features"
)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

X_train_hand.to_parquet(
    FEATURE_DIR / "train_handcrafted.parquet"
)
X_valid_hand.to_parquet(
    FEATURE_DIR / "valid_handcrafted.parquet"
)
X_test_hand.to_parquet(
    FEATURE_DIR / "test_handcrafted.parquet"
)

print("Handcrafted features saved successfully.")

Handcrafted features saved successfully.
